# Prepare one 4D-CT patient and train canonical MedGS models

This notebook prepares one respiratory 4D-CT study and trains static MedGS models for one selected respiratory phase.

It:

1. selects a patient with `PATIENT_ID`;
2. unpacks that patient's DICOM series;
3. lists available studies and uses the manually selected `STUDY_INSTANCE_UID`;
4. identifies and orders the respiratory CT phases;
5. saves every phase as raw and Gaussian-denoised HU volumes;
6. provides an interactive original/denoised slice browser;
7. converts one canonical phase to the MedGS `original/` and `mirror/` layout;
8. trains and evaluates raw and denoised canonical models;
9. reports PSNR, SSIM, and LPIPS and provides an input/reconstruction browser.

The canonical model represents only one static respiratory phase. Respiratory deformation, temporal conditioning, and multi-phase training are added later in this notebook.

## How to use it

The notebook may be opened locally in PyCharm while running on a remote WORF or Helios kernel. All paths refer to the filesystem of the remote kernel.

For a normal run:

1. set `PATIENT_ID` and the training parameters;
2. unpack the selected patient;
3. inspect the study table;
4. assign the chosen full DICOM UID to `STUDY_INSTANCE_UID`;
5. run the remaining cells from top to bottom.

Prepared volumes, MedGS datasets, and completed models are reused unless the corresponding rebuild flags are enabled.

The Gaussian filter is only a simple raw-versus-denoised baseline, not a final medical denoising method.

## Outputs

```text
results/
├── prepared_4d_patients/
│   └── <PATIENT_ID>/
│       └── study_<UID_SUFFIX>/
│           ├── preparation.json
│           ├── phase_summary.csv
│           ├── phase_slice_manifest.csv
│           └── volumes/
│               ├── raw/
│               │   ├── phase_00.npy
│               │   └── ...
│               └── denoised/
│                   ├── phase_00.npy
│                   └── ...
└── canonical_medgs/
    └── patient_<PATIENT_ID>__study_<UID_SUFFIX>__phase_<PHASE>__<raw|denoised>__poly<P>__iter<N>/
        ├── run.json
        ├── frame_manifest.csv
        ├── dataset/
        │   ├── original/
        │   └── mirror/
        ├── model/
        ├── train_metrics.csv
        └── train_metrics_summary.csv
```

`<UID_SUFFIX>` is the last 12 digits of the full `StudyInstanceUID`. The complete UID is retained in the JSON and CSV metadata.

`prepared_4d_patients/` contains reusable study-level medical data. `canonical_medgs/` contains separate training experiments for different phases, input representations, and MedGS settings.


In [17]:
# Main configuration: normally this is the only cell that needs editing.

PATIENT_ID = "117_HM10395"

# Leave as None to select automatically the study with the most CT phases.
# STUDY_INSTANCE_UID = None

# The phase used to train the canonical MedGS model.
CANONICAL_PHASE_PERCENT = 20.0

# Choose which prepared representation is converted to MedGS PNG input.
TRAIN_ON_DENOISED = False

HU_WINDOW_LOW = -1000.0
HU_WINDOW_HIGH = 400.0

# Mild 3D Gaussian denoising in voxel units: (slice, row, column).
# DENOISE_SIGMA = (0.35, 0.70, 0.70)
DENOISE_SIGMA = (0.20, 0.40, 0.40)

ITERATIONS = 30_000
POLY_DEGREE = 2
BATCH_SIZE = 3
CAMERA = "mirror"

RUN_TRAINING = True
REUSE_COMPLETED_MODEL = True
REBUILD_PREPARED_VOLUMES = False
REBUILD_MEDGS_DATASET = False

## 1. Imports and project paths

Shared DICOM data and the MedGS repository are read from the existing project location. Generated arrays, PNG files, manifests, and model checkpoints are written under the current user's `medgs4d/results` directory.

In [ ]:
from __future__ import annotations

from pathlib import Path

import ipywidgets as widgets
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydicom
import re
import shlex
import subprocess
import sys
import torch
from IPython.display import clear_output, display
from PIL import Image
from pydicom.dataset import Dataset
from scipy.ndimage import gaussian_filter
from typing import Any

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 500)

# ------------------------------------------------------------------
# Project paths
# ------------------------------------------------------------------

# WORF paths — active.
PROJECT_ROOT = Path(
    "/home/jovyan/shared/mtm_medgs_stack"
).resolve()

DATA_ROOT = (
        PROJECT_ROOT
        / "data"
        / "tcia_4d_lung"
)

SERIES_ZIPS_ROOT = (
        DATA_ROOT
        / "raw"
        / "series_zips"
)

DICOM_ROOT = (
        DATA_ROOT
        / "raw"
        / "dicom_by_series"
)

MEDGS4D_REPOSITORY = (
        PROJECT_ROOT
        / "repo"
        / "medgs4d"
)

MEDGS_REPOSITORY = (
        PROJECT_ROOT
        / "repo"
        / "MedGS"
)

RESULTS_ROOT = PROJECT_ROOT / "results"

# HELIOS paths — uncomment this block and comment out the WORF block
# when running the notebook on Helios.
#
# PROJECT_ROOT = Path(
#     "/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d"
# ).resolve()
#
# DATA_ROOT = PROJECT_ROOT / "data"
#
# SERIES_ZIPS_ROOT = (
#     DATA_ROOT
#     / "raw"
#     / "series_zips"
# )
#
# DICOM_ROOT = (
#     DATA_ROOT
#     / "raw"
#     / "dicom_by_series"
# )
#
# MEDGS4D_REPOSITORY = (
#     PROJECT_ROOT
#     / "repo"
#     / "medgs4d"
# )
#
# MEDGS_REPOSITORY = (
#     PROJECT_ROOT
#     / "repo"
#     / "MedGS"
# )
#
# RESULTS_ROOT = PROJECT_ROOT / "results"


PREPARED_4D_ROOT = (
        RESULTS_ROOT
        / "prepared_4d_patients"
)

CANONICAL_EXPERIMENTS_ROOT = (
        RESULTS_ROOT
        / "canonical_medgs"
)

required_paths = [DICOM_ROOT, MEDGS_REPOSITORY]
missing_paths = [path for path in required_paths if not path.is_dir()]
if missing_paths:
    raise FileNotFoundError(
        "Required directories are missing:\n"
        + "\n".join(str(path) for path in missing_paths)
    )



In [3]:
required_paths = [
    SERIES_ZIPS_ROOT,
    MEDGS4D_REPOSITORY,
    MEDGS_REPOSITORY,
]

missing_paths = [
    path
    for path in required_paths
    if not path.is_dir()
]

if missing_paths:
    raise FileNotFoundError(
        "Required directories are missing:\n"
        + "\n".join(str(path) for path in missing_paths)
    )

DICOM_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
PREPARED_4D_ROOT.mkdir(parents=True, exist_ok=True)
CANONICAL_EXPERIMENTS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"Python:           {sys.executable}")
print(f"CUDA available:   {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU:              {torch.cuda.get_device_name(0)}")

print(f"Project root:     {PROJECT_ROOT}")
print(f"Series ZIPs:      {SERIES_ZIPS_ROOT}")
print(f"DICOM root:       {DICOM_ROOT}")
print(f"MedGS4D repo:     {MEDGS4D_REPOSITORY}")
print(f"MedGS repository: {MEDGS_REPOSITORY}")
print(f"Results root:     {RESULTS_ROOT}")

Python:           /home/jovyan/shared/mtm_medgs_stack/envs/medgs-worf/bin/python
CUDA available:   True
GPU:              NVIDIA RTX A5500
Project root:     /home/jovyan/shared/mtm_medgs_stack
Series ZIPs:      /home/jovyan/shared/mtm_medgs_stack/data/tcia_4d_lung/raw/series_zips
DICOM root:       /home/jovyan/shared/mtm_medgs_stack/data/tcia_4d_lung/raw/dicom_by_series
MedGS4D repo:     /home/jovyan/shared/mtm_medgs_stack/repo/medgs4d
MedGS repository: /home/jovyan/shared/mtm_medgs_stack/repo/MedGS
Results root:     /home/jovyan/shared/mtm_medgs_stack/results


In [6]:
patient_ids = sorted(
    path.name
    for path in SERIES_ZIPS_ROOT.iterdir()
    if path.is_dir()
)

print(f"Available patients: {len(patient_ids)}\n")

for patient_id in patient_ids:
    marker = "  <-- selected" if patient_id == PATIENT_ID else ""
    print(f"{patient_id}{marker}")

Available patients: 20

100_HM10395
101_HM10395
102_HM10395
103_HM10395
104_HM10395
105_HM10395
106_HM10395
107_HM10395
108_HM10395
109_HM10395
110_HM10395
111_HM10395
112_HM10395
113_HM10395
114_HM10395
115_HM10395
116_HM10395
117_HM10395  <-- selected
118_HM10395
119_HM10395


In [22]:
import shutil

dataset_readers_path = (
        MEDGS_REPOSITORY
        / "models"
        / "scenes"
        / "dataset_readers.py"
)

source = dataset_readers_path.read_text(
    encoding="utf-8"
)

assert "dtype=np.byte" in source

source = source.replace(
    "dtype=np.byte",
    "dtype=np.uint8",
)

dataset_readers_path.write_text(
    source,
    encoding="utf-8",
)

# shutil.rmtree(model_root)

print(f"Patched MedGS reader: {dataset_readers_path}")
print(f"Removed failed model: {model_root}")

Patched MedGS reader: /home/jovyan/shared/mtm_medgs_stack/repo/MedGS/models/scenes/dataset_readers.py
Removed failed model: /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/model


## 2. DICOM helper functions

Respiratory phase percentages are read from `SeriesDescription`, exactly as in the earlier inspection notebook. Slices are ordered by their physical coordinate along the DICOM slice normal, not by filename.

In [5]:
# Extract the selected patient using the existing MedGS4D
# extraction and verification functions.

from concurrent.futures import ThreadPoolExecutor
import importlib.util

UNPACK_SCRIPT = (
        MEDGS4D_REPOSITORY
        / "scripts"
        / "unpack_4d_lung.py"
)

if not UNPACK_SCRIPT.is_file():
    raise FileNotFoundError(
        f"Unpacking script not found: {UNPACK_SCRIPT}"
    )

spec = importlib.util.spec_from_file_location(
    "medgs4d_unpack_4d_lung",
    UNPACK_SCRIPT,
)

if spec is None or spec.loader is None:
    raise RuntimeError(
        f"Could not load the unpacking script: {UNPACK_SCRIPT}"
    )

unpack_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(unpack_module)

archives = unpack_module.discover_archives(
    SERIES_ZIPS_ROOT,
    [PATIENT_ID],
)

if not archives:
    raise FileNotFoundError(
        f"No ZIP archives found for patient {PATIENT_ID} "
        f"under {SERIES_ZIPS_ROOT}"
    )

tasks = [
    (
        str(archive),
        str(SERIES_ZIPS_ROOT),
        str(DICOM_ROOT),
        False,  # force
        False,  # dry_run
    )
    for archive in archives
]

print(f"Patient:             {PATIENT_ID}")
print(f"Selected archives:   {len(archives)}")
print(f"Archive root:        {SERIES_ZIPS_ROOT}")
print(f"Destination root:    {DICOM_ROOT}")
print("Extracting and verifying series...")

worker_count = min(4, len(tasks))

with ThreadPoolExecutor(
        max_workers=worker_count
) as executor:
    results = list(
        executor.map(
            unpack_module.extract_one,
            tasks,
        )
    )

status_counts = {}

for result in results:
    status = result["status"]
    status_counts[status] = (
            status_counts.get(status, 0) + 1
    )

print("\nExtraction summary:")

for status in (
        "extracted",
        "skipped",
        "failed",
):
    print(
        f"  {status.capitalize():10s}: "
        f"{status_counts.get(status, 0)}"
    )

failed_results = [
    result
    for result in results
    if result["status"] == "failed"
]

if failed_results:
    failed_preview = "\n".join(
        (
            f"{result['archive']}: "
            f"{result['message']}"
        )
        for result in failed_results[:10]
    )

    raise RuntimeError(
        "Some archives could not be extracted:\n"
        + failed_preview
    )

patient_dicom_root = DICOM_ROOT / PATIENT_ID

extracted_series = sorted(
    path
    for path in patient_dicom_root.iterdir()
    if path.is_dir()
    and not path.name.startswith(".")
)

print(
    f"\nPatient {PATIENT_ID}: "
    f"{len(extracted_series)} extracted series."
)
print(f"Patient DICOM root: {patient_dicom_root}")

Patient:             117_HM10395
Selected archives:   420
Archive root:        /home/jovyan/shared/mtm_medgs_stack/data/tcia_4d_lung/raw/series_zips
Destination root:    /home/jovyan/shared/mtm_medgs_stack/data/tcia_4d_lung/raw/dicom_by_series
Extracting and verifying series...

Extraction summary:
  Extracted : 420
  Skipped   : 0
  Failed    : 0

Patient 117_HM10395: 420 extracted series.
Patient DICOM root: /home/jovyan/shared/mtm_medgs_stack/data/tcia_4d_lung/raw/dicom_by_series/117_HM10395


In [7]:
PHASE_PATTERN = re.compile(r"(\d+(?:\.\d+)?)\s*%")


def parse_phase_percent(description: str) -> float:
    """Extract a respiratory phase percentage or return NaN."""

    match = PHASE_PATTERN.search(description)
    return float(match.group(1)) if match else float("nan")


def phase_tag(phase: float) -> str:
    """Return a filesystem-safe phase label."""

    return f"{phase:g}".replace(".", "p")


def list_dicom_files(series_dir: Path) -> list[Path]:
    """Return DICOM files stored directly in one series directory."""

    return sorted(path for path in series_dir.glob("*.dcm") if path.is_file())


def read_float_vector(
        dataset: Dataset,
        attribute: str,
        expected_length: int,
) -> np.ndarray:
    """Read a required fixed-length numeric DICOM vector."""

    if not hasattr(dataset, attribute):
        raise ValueError(f"Missing DICOM attribute {attribute}.")

    vector = np.asarray(
        [float(value) for value in getattr(dataset, attribute)],
        dtype=np.float64,
    )
    if vector.size != expected_length:
        raise ValueError(
            f"{attribute} has length {vector.size}; expected {expected_length}."
        )
    return vector


def slice_coordinate(dataset: Dataset) -> float:
    """Calculate the signed position along the DICOM slice normal."""

    position = read_float_vector(dataset, "ImagePositionPatient", 3)
    orientation = read_float_vector(dataset, "ImageOrientationPatient", 6)
    normal = np.cross(orientation[:3], orientation[3:])
    normal /= np.linalg.norm(normal)
    return float(np.dot(position, normal))


def inspect_series(patient_directory: Path) -> pd.DataFrame:
    """Read one representative header from every patient series."""

    records: list[dict[str, Any]] = []
    for series_dir in sorted(path for path in patient_directory.iterdir() if path.is_dir()):
        files = list_dicom_files(series_dir)
        if not files:
            continue

        dataset = pydicom.dcmread(
            str(files[0]), stop_before_pixels=True, force=True
        )
        description = str(getattr(dataset, "SeriesDescription", ""))
        records.append(
            {
                "StudyDate": str(getattr(dataset, "StudyDate", "")),
                "StudyDescription": str(
                    getattr(dataset, "StudyDescription", "")
                ),
                "StudyInstanceUID": str(
                    getattr(dataset, "StudyInstanceUID", "")
                ),
                "Modality": str(getattr(dataset, "Modality", "")),
                "PhasePercent": parse_phase_percent(description),
                "SeriesNumber": getattr(dataset, "SeriesNumber", ""),
                "SeriesDescription": description,
                "SeriesInstanceUID": str(
                    getattr(dataset, "SeriesInstanceUID", "")
                ),
                "DICOM files": len(files),
                "Series path": str(series_dir),
            }
        )

    frame = pd.DataFrame(records)
    if frame.empty:
        raise RuntimeError(f"No readable DICOM series found in {patient_directory}.")
    return frame


def read_slice_table(
        series_dir: Path,
        expected_series_uid: str,
) -> pd.DataFrame:
    """Read and physically order all CT slices from one series."""

    records: list[dict[str, Any]] = []
    for path in list_dicom_files(series_dir):
        dataset = pydicom.dcmread(
            str(path), stop_before_pixels=True, force=True
        )
        if str(getattr(dataset, "Modality", "")) != "CT":
            raise ValueError(f"Non-CT object found: {path}")
        if str(getattr(dataset, "SeriesInstanceUID", "")) != expected_series_uid:
            raise ValueError(f"Unexpected SeriesInstanceUID in {path}")

        position = read_float_vector(dataset, "ImagePositionPatient", 3)
        orientation = read_float_vector(
            dataset, "ImageOrientationPatient", 6
        )
        pixel_spacing = read_float_vector(dataset, "PixelSpacing", 2)

        records.append(
            {
                "File": path.name,
                "Path": str(path),
                "SOPInstanceUID": str(
                    getattr(dataset, "SOPInstanceUID", "")
                ),
                "InstanceNumber": int(
                    getattr(dataset, "InstanceNumber", -1)
                ),
                "SliceCoordinate": slice_coordinate(dataset),
                "PositionX": float(position[0]),
                "PositionY": float(position[1]),
                "PositionZ": float(position[2]),
                "Rows": int(dataset.Rows),
                "Columns": int(dataset.Columns),
                "PixelSpacingRow": float(pixel_spacing[0]),
                "PixelSpacingColumn": float(pixel_spacing[1]),
                "SliceThickness": float(
                    getattr(dataset, "SliceThickness", np.nan)
                ),
                "RescaleSlope": float(
                    getattr(dataset, "RescaleSlope", 1.0)
                ),
                "RescaleIntercept": float(
                    getattr(dataset, "RescaleIntercept", 0.0)
                ),
                "Orientation": tuple(float(value) for value in orientation),
            }
        )

    frame = pd.DataFrame(records).sort_values(
        ["SliceCoordinate", "InstanceNumber", "File"]
    ).reset_index(drop=True)
    frame.insert(0, "SliceIndex", np.arange(len(frame), dtype=int))
    return frame


def read_hu_image(path: Path) -> np.ndarray:
    """Read one CT slice and convert stored values to Hounsfield units."""

    dataset = pydicom.dcmread(str(path), force=True)
    pixels = dataset.pixel_array.astype(np.float32)
    slope = float(getattr(dataset, "RescaleSlope", 1.0))
    intercept = float(getattr(dataset, "RescaleIntercept", 0.0))
    return pixels * slope + intercept


def window_hu(image_hu: np.ndarray, low: float, high: float) -> np.ndarray:
    """Clip one HU image and normalize it to [0, 1]."""

    if high <= low:
        raise ValueError("HU_WINDOW_HIGH must exceed HU_WINDOW_LOW.")
    clipped = np.clip(image_hu, low, high)
    return (clipped - low) / (high - low)


def grayscale_to_rgb_uint8(image: np.ndarray) -> np.ndarray:
    """Convert normalized grayscale to three-channel uint8 RGB."""

    grayscale = np.round(image * 255.0).astype(np.uint8)
    return np.repeat(grayscale[..., None], 3, axis=2)

## 3. Select the patient, study, and one CT series per phase

If multiple studies exist for the patient, automatic selection prefers the study with the largest number of distinct respiratory phases and then the largest total number of CT slices. Within a phase, the CT series with the largest slice count is selected.

In [10]:
patient_dir = DICOM_ROOT / PATIENT_ID

series_df = inspect_series(patient_dir)

ct_phase_series_df = series_df.loc[
    (series_df["Modality"] == "CT")
    & series_df["PhasePercent"].notna()
    ].copy()

if ct_phase_series_df.empty:
    raise RuntimeError(
        f"No phase-labelled CT series found for patient {PATIENT_ID}."
    )

study_summary_df = (
    ct_phase_series_df.groupby(
        "StudyInstanceUID",
        as_index=False,
    )
    .agg(
        StudyDate=("StudyDate", "first"),
        StudyDescription=("StudyDescription", "first"),
        Phases=("PhasePercent", "nunique"),
        CTSeries=("SeriesInstanceUID", "nunique"),
        TotalSlices=("DICOM files", "sum"),
    )
    .sort_values(
        ["Phases", "TotalSlices"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

study_summary_df.insert(
    0,
    "StudyIndex",
    study_summary_df.index,
)

print(f"Patient:          {PATIENT_ID}")
print(f"Detected studies: {len(study_summary_df)}")
print()
# print(study_summary_df.to_string(index=False))

display(study_summary_df)

Patient:          117_HM10395
Detected studies: 34



,StudyIndex,StudyInstanceUID,StudyDate,StudyDescription,Phases,CTSeries,TotalSlices
0,0,1.3.6.1.4.1.14519.5.2.1.6834.5010.378204929111417980831212264180,20001024,p4,10,10,1470
1,1,1.3.6.1.4.1.14519.5.2.1.6834.5010.262507920541737267747764039873,20001031,p4,10,10,1170
2,2,1.3.6.1.4.1.14519.5.2.1.6834.5010.292491295549436721879304177874,20001107,p4,10,10,1170
3,3,1.3.6.1.4.1.14519.5.2.1.6834.5010.158466884027749484040796826752,20001128,p4,10,10,1050
4,4,1.3.6.1.4.1.14519.5.2.1.6834.5010.154051254217750840976295179553,20001121,p4,10,10,910
5,5,1.3.6.1.4.1.14519.5.2.1.6834.5010.322681136408267760646893201309,20001204,p4,10,10,900
6,6,1.3.6.1.4.1.14519.5.2.1.6834.5010.183612837156574038288724943681,20001117,p4,10,10,880
7,7,1.3.6.1.4.1.14519.5.2.1.6834.5010.253771392853795552765799739432,20001009,p4,10,10,770
8,8,1.3.6.1.4.1.14519.5.2.1.6834.5010.107025610280264464701477361897,20001019,p4,10,10,500
9,9,1.3.6.1.4.1.14519.5.2.1.6834.5010.115679706801800900578804691850,20001106,p4,10,10,500


In [24]:
# Select one StudyInstanceUID from the table above.
STUDY_INSTANCE_UID = (
    "1.3.6.1.4.1.14519.5.2.1.6834.5010."
    "378204929111417980831212264180"
)

In [12]:

selected_study_uid = STUDY_INSTANCE_UID

selected_study_series_df = ct_phase_series_df.loc[
    ct_phase_series_df["StudyInstanceUID"]
    == selected_study_uid
    ].copy()

if selected_study_series_df.empty:
    raise ValueError(
        f"Study not found for patient {PATIENT_ID}: "
        f"{selected_study_uid}"
    )

phase_series_df = (
    selected_study_series_df.sort_values(
        [
            "PhasePercent",
            "DICOM files",
            "SeriesInstanceUID",
        ],
        ascending=[True, False, True],
    )
    .drop_duplicates(
        subset=["PhasePercent"],
        keep="first",
    )
    .sort_values("PhasePercent")
    .reset_index(drop=True)
)

selected_study_info = study_summary_df.loc[
    study_summary_df["StudyInstanceUID"]
    == selected_study_uid
    ].iloc[0]

phase_summary_df = phase_series_df[
    [
        "PhasePercent",
        "DICOM files",
        "SeriesNumber",
        "SeriesDescription",
        "SeriesInstanceUID",
        "Series path",
    ]
].copy()

print(f"Selected patient: {PATIENT_ID}")
print(f"Selected study:   {selected_study_uid}")
print(f"Study date:       {selected_study_info['StudyDate']}")
print(f"Detected phases:  {len(phase_series_df)}")
print(f"Total slices:     {selected_study_info['TotalSlices']}")
print()
# print(phase_summary_df.to_string(index=False))

display(phase_summary_df)

Selected patient: 117_HM10395
Selected study:   1.3.6.1.4.1.14519.5.2.1.6834.5010.378204929111417980831212264180
Study date:       20001024
Detected phases:  10
Total slices:     1470



,PhasePercent,DICOM files,SeriesNumber,SeriesDescription,SeriesInstanceUID,Series path
0,0.0,147,1,"P4^P117^S301^I00003, Gated, 0.0%A",1.3.6.1.4.1.14519.5.2.1.6834.5010.148529694733361305012151030542,/home/jovyan/shared/mtm_medgs_stack/data/tcia_4d_lung/raw/dicom_by_series/117_HM10395/1.3.6.1.4.1.14519.5.2.1.6834.5...
1,10.0,147,1,"P4^P117^S301^I00004, Gated, 10.0%A",1.3.6.1.4.1.14519.5.2.1.6834.5010.126969234347231214775450287756,/home/jovyan/shared/mtm_medgs_stack/data/tcia_4d_lung/raw/dicom_by_series/117_HM10395/1.3.6.1.4.1.14519.5.2.1.6834.5...
2,20.0,147,1,"P4^P117^S301^I00005, Gated, 20.0%A",1.3.6.1.4.1.14519.5.2.1.6834.5010.604396030201226976476677184587,/home/jovyan/shared/mtm_medgs_stack/data/tcia_4d_lung/raw/dicom_by_series/117_HM10395/1.3.6.1.4.1.14519.5.2.1.6834.5...
3,30.0,147,1,"P4^P117^S301^I00006, Gated, 30.0%A",1.3.6.1.4.1.14519.5.2.1.6834.5010.265853994490570208708415104191,/home/jovyan/shared/mtm_medgs_stack/data/tcia_4d_lung/raw/dicom_by_series/117_HM10395/1.3.6.1.4.1.14519.5.2.1.6834.5...
4,40.0,147,1,"P4^P117^S301^I00007, Gated, 40.0%A",1.3.6.1.4.1.14519.5.2.1.6834.5010.268641441554830768026549171437,/home/jovyan/shared/mtm_medgs_stack/data/tcia_4d_lung/raw/dicom_by_series/117_HM10395/1.3.6.1.4.1.14519.5.2.1.6834.5...
5,50.0,147,1,"P4^P117^S301^I00008, Gated, 50.0%A",1.3.6.1.4.1.14519.5.2.1.6834.5010.142000484786251122043719870687,/home/jovyan/shared/mtm_medgs_stack/data/tcia_4d_lung/raw/dicom_by_series/117_HM10395/1.3.6.1.4.1.14519.5.2.1.6834.5...
6,60.0,147,1,"P4^P117^S301^I00009, Gated, 60.0%A",1.3.6.1.4.1.14519.5.2.1.6834.5010.183882293876244172928996511379,/home/jovyan/shared/mtm_medgs_stack/data/tcia_4d_lung/raw/dicom_by_series/117_HM10395/1.3.6.1.4.1.14519.5.2.1.6834.5...
7,70.0,147,1,"P4^P117^S301^I00010, Gated, 70.0%A",1.3.6.1.4.1.14519.5.2.1.6834.5010.240633073422000755437203884091,/home/jovyan/shared/mtm_medgs_stack/data/tcia_4d_lung/raw/dicom_by_series/117_HM10395/1.3.6.1.4.1.14519.5.2.1.6834.5...
8,80.0,147,1,"P4^P117^S301^I00011, Gated, 80.0%A",1.3.6.1.4.1.14519.5.2.1.6834.5010.213548587798491983341081227810,/home/jovyan/shared/mtm_medgs_stack/data/tcia_4d_lung/raw/dicom_by_series/117_HM10395/1.3.6.1.4.1.14519.5.2.1.6834.5...
9,90.0,147,1,"P4^P117^S301^I00012, Gated, 90.0%A",1.3.6.1.4.1.14519.5.2.1.6834.5010.329406950451472064474907839203,/home/jovyan/shared/mtm_medgs_stack/data/tcia_4d_lung/raw/dicom_by_series/117_HM10395/1.3.6.1.4.1.14519.5.2.1.6834.5...


## 4. Read and summarize all phase stacks

Each phase is ordered independently by physical slice position. The summary below shows whether the phase stacks have matching image geometry and comparable coordinate ranges. No resampling is performed in this simple preparation notebook.

In [14]:
assert phase_series_df["StudyInstanceUID"].astype(str).eq(
    str(STUDY_INSTANCE_UID)
).all()

phase_slice_tables: dict[float, pd.DataFrame] = {}
phase_summary_rows = []

for _, phase_row in phase_series_df.iterrows():
    phase = float(phase_row["PhasePercent"])

    table = read_slice_table(
        Path(phase_row["Series path"]),
        str(phase_row["SeriesInstanceUID"]),
    )

    coordinates = table["SliceCoordinate"].to_numpy(dtype=float)
    spacings = np.diff(coordinates)

    assert len(table) >= 3
    assert np.all(spacings > 0)

    phase_slice_tables[phase] = table

    phase_summary_rows.append(
        {
            "PhasePercent": phase,
            "Slices": len(table),
            "Rows": int(table.iloc[0]["Rows"]),
            "Columns": int(table.iloc[0]["Columns"]),
            "FirstCoordinate": float(coordinates[0]),
            "LastCoordinate": float(coordinates[-1]),
            "MedianSpacing": float(np.median(spacings)),
            "PixelSpacingRow": float(
                table.iloc[0]["PixelSpacingRow"]
            ),
            "PixelSpacingColumn": float(
                table.iloc[0]["PixelSpacingColumn"]
            ),
        }
    )

phase_summary_df = (
    pd.DataFrame(phase_summary_rows)
    .sort_values("PhasePercent")
    .reset_index(drop=True)
)

print(f"Patient:            {PATIENT_ID}")
print(f"Study:              {STUDY_INSTANCE_UID}")
print(f"Respiratory phases: {len(phase_summary_df)}")
print(f"Total CT slices:    {int(phase_summary_df['Slices'].sum())}")
print(
    "Slices per phase:  "
    f"min={int(phase_summary_df['Slices'].min())}, "
    f"max={int(phase_summary_df['Slices'].max())}"
)

display(phase_summary_df)

Patient:            117_HM10395
Study:              1.3.6.1.4.1.14519.5.2.1.6834.5010.378204929111417980831212264180
Respiratory phases: 10
Total CT slices:    1470
Slices per phase:  min=147, max=147


,PhasePercent,Slices,Rows,Columns,FirstCoordinate,LastCoordinate,MedianSpacing,PixelSpacingRow,PixelSpacingColumn
0,0.0,147,512,512,-266.2,171.8,3.0,0.9766,0.9766
1,10.0,147,512,512,-266.2,171.8,3.0,0.9766,0.9766
2,20.0,147,512,512,-266.2,171.8,3.0,0.9766,0.9766
3,30.0,147,512,512,-266.2,171.8,3.0,0.9766,0.9766
4,40.0,147,512,512,-266.2,171.8,3.0,0.9766,0.9766
5,50.0,147,512,512,-266.2,171.8,3.0,0.9766,0.9766
6,60.0,147,512,512,-266.2,171.8,3.0,0.9766,0.9766
7,70.0,147,512,512,-266.2,171.8,3.0,0.9766,0.9766
8,80.0,147,512,512,-266.2,171.8,3.0,0.9766,0.9766
9,90.0,147,512,512,-266.2,171.8,3.0,0.9766,0.9766


## 5. Prepare original and mildly denoised HU volumes

For every phase, the notebook writes two NumPy arrays:

- `raw/phase_*.npy`: original HU values read from DICOM;
- `denoised/phase_*.npy`: the same volume after a mild 3D Gaussian filter.

Volumes are processed one phase at a time, so the complete 4D dataset is not held twice in RAM. The `.npy` format also allows memory-mapped browsing without loading all phases at once.

In [18]:
study_tag = str(STUDY_INSTANCE_UID)[-12:]

patient_prepared_root = (
        PREPARED_4D_ROOT
        / PATIENT_ID
        / f"study_{study_tag}"
)

raw_volumes_root = patient_prepared_root / "volumes" / "raw"
denoised_volumes_root = (
        patient_prepared_root
        / "volumes"
        / "denoised"
)

raw_volumes_root.mkdir(parents=True, exist_ok=True)
denoised_volumes_root.mkdir(parents=True, exist_ok=True)

phase_volume_paths: dict[tuple[float, bool], Path] = {}
manifest_parts = []

for phase, table in phase_slice_tables.items():
    tag = phase_tag(phase)

    raw_path = raw_volumes_root / f"phase_{tag}.npy"
    denoised_path = (
            denoised_volumes_root
            / f"phase_{tag}.npy"
    )

    rebuild_raw = (
            REBUILD_PREPARED_VOLUMES
            or not raw_path.is_file()
    )

    if rebuild_raw:
        print(f"Reading phase {phase:g}%: {len(table)} slices")

        raw_volume = np.stack(
            [
                read_hu_image(Path(path))
                for path in table["Path"]
            ],
            axis=0,
        ).astype(np.float32)

        np.save(raw_path, raw_volume)

    else:
        raw_volume = np.load(
            raw_path,
            mmap_mode="r",
        )

    rebuild_denoised = (
            REBUILD_PREPARED_VOLUMES
            or not denoised_path.is_file()
    )

    if rebuild_denoised:
        print(
            f"Denoising phase {phase:g}% "
            f"with sigma={DENOISE_SIGMA}"
        )

        denoised_volume = gaussian_filter(
            np.asarray(
                raw_volume,
                dtype=np.float32,
            ),
            sigma=DENOISE_SIGMA,
            mode="nearest",
        ).astype(np.float32)

        np.save(
            denoised_path,
            denoised_volume,
        )

    phase_volume_paths[(phase, False)] = raw_path
    phase_volume_paths[(phase, True)] = denoised_path

    phase_manifest = table.copy()
    phase_manifest.insert(
        0,
        "PhasePercent",
        phase,
    )

    phase_manifest["StudyInstanceUID"] = str(
        STUDY_INSTANCE_UID
    )
    phase_manifest["RawVolumePath"] = str(raw_path)
    phase_manifest["DenoisedVolumePath"] = str(
        denoised_path
    )

    manifest_parts.append(phase_manifest)

all_slices_manifest_df = (
    pd.concat(
        manifest_parts,
        ignore_index=True,
    )
    .sort_values(
        ["PhasePercent", "SliceIndex"]
    )
    .reset_index(drop=True)
)

manifest_path = (
        patient_prepared_root
        / "phase_slice_manifest.csv"
)

summary_path = (
        patient_prepared_root
        / "phase_summary.csv"
)

config_path = (
        patient_prepared_root
        / "preparation.json"
)

all_slices_manifest_df.to_csv(
    manifest_path,
    index=False,
)

phase_summary_df.to_csv(
    summary_path,
    index=False,
)

preparation_config = {
    "patient_id": PATIENT_ID,
    "study_instance_uid": str(STUDY_INSTANCE_UID),
    "phases": sorted(
        float(value)
        for value in phase_slice_tables
    ),
    "hu_window": [
        HU_WINDOW_LOW,
        HU_WINDOW_HIGH,
    ],
    "denoise_method": "scipy.ndimage.gaussian_filter",
    "denoise_sigma_voxels": list(DENOISE_SIGMA),
    "raw_volumes_root": str(raw_volumes_root),
    "denoised_volumes_root": str(
        denoised_volumes_root
    ),
    "manifest": str(manifest_path),
}

config_path.write_text(
    json.dumps(
        preparation_config,
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)

print(f"Patient:               {PATIENT_ID}")
print(f"Study:                 {STUDY_INSTANCE_UID}")
print(f"Prepared patient root: {patient_prepared_root}")
print(f"Manifest:              {manifest_path}")
print(f"Raw volumes:           {raw_volumes_root}")
print(f"Denoised volumes:      {denoised_volumes_root}")

Reading phase 0%: 147 slices
Denoising phase 0% with sigma=(0.2, 0.4, 0.4)
Reading phase 10%: 147 slices
Denoising phase 10% with sigma=(0.2, 0.4, 0.4)
Reading phase 20%: 147 slices
Denoising phase 20% with sigma=(0.2, 0.4, 0.4)
Reading phase 30%: 147 slices
Denoising phase 30% with sigma=(0.2, 0.4, 0.4)
Reading phase 40%: 147 slices
Denoising phase 40% with sigma=(0.2, 0.4, 0.4)
Reading phase 50%: 147 slices
Denoising phase 50% with sigma=(0.2, 0.4, 0.4)
Reading phase 60%: 147 slices
Denoising phase 60% with sigma=(0.2, 0.4, 0.4)
Reading phase 70%: 147 slices
Denoising phase 70% with sigma=(0.2, 0.4, 0.4)
Reading phase 80%: 147 slices
Denoising phase 80% with sigma=(0.2, 0.4, 0.4)
Reading phase 90%: 147 slices
Denoising phase 90% with sigma=(0.2, 0.4, 0.4)
Patient:               117_HM10395
Study:                 None
Prepared patient root: /home/jovyan/shared/mtm_medgs_stack/results/prepared_4d_patients/117_HM10395/study_None
Manifest:              /home/jovyan/shared/mtm_medgs_stack

## 6. Interactive phase and slice browser

Use the phase slider to move through respiratory time, the slice slider to move through the ordered CT stack, and the checkbox to switch between the original and denoised volume.

In [19]:
available_phases = sorted(phase_slice_tables)
initial_phase = float(CANONICAL_PHASE_PERCENT)

phase_slider = widgets.SelectionSlider(
    options=[
        (f"{phase:g}%", phase)
        for phase in available_phases
    ],
    value=initial_phase,
    description="Phase",
    continuous_update=False,
    layout=widgets.Layout(width="650px"),
)

slice_slider = widgets.IntSlider(
    value=len(phase_slice_tables[initial_phase]) // 2,
    min=0,
    max=len(phase_slice_tables[initial_phase]) - 1,
    step=1,
    description="Slice",
    continuous_update=False,
    layout=widgets.Layout(width="650px"),
)

denoised_checkbox = widgets.Checkbox(
    value=False,
    description="Show denoised",
    indent=False,
)

browser_output = widgets.Output()


def update_slice_range(change: dict | None = None) -> None:
    """Update the valid slice range after changing the phase."""

    phase = float(phase_slider.value)
    maximum = len(phase_slice_tables[phase]) - 1

    slice_slider.max = maximum
    slice_slider.value = min(
        slice_slider.value,
        maximum,
    )


def draw_selected_slice(change: dict | None = None) -> None:
    """Render the selected CT slice."""

    phase = float(phase_slider.value)
    slice_index = int(slice_slider.value)
    use_denoised = bool(denoised_checkbox.value)

    volume = np.load(
        phase_volume_paths[(phase, use_denoised)],
        mmap_mode="r",
    )

    table = phase_slice_tables[phase]
    row = table.iloc[slice_index]

    image = window_hu(
        np.asarray(volume[slice_index]),
        HU_WINDOW_LOW,
        HU_WINDOW_HIGH,
    )

    representation = (
        "denoised"
        if use_denoised
        else "original"
    )

    with browser_output:
        clear_output(wait=True)

        plt.figure(figsize=(7, 7))
        plt.imshow(
            image,
            cmap="gray",
            vmin=0.0,
            vmax=1.0,
        )

        plt.title(
            f"Patient {PATIENT_ID} | "
            f"phase {phase:g}% | "
            f"slice {slice_index}/{len(table) - 1}\n"
            f"{representation} | "
            f"coordinate "
            f"{row['SliceCoordinate']:.3f} mm | "
            f"HU [{HU_WINDOW_LOW:g}, {HU_WINDOW_HIGH:g}]"
        )

        plt.axis("off")
        plt.tight_layout()
        plt.show()


phase_slider.observe(
    update_slice_range,
    names="value",
)

phase_slider.observe(
    draw_selected_slice,
    names="value",
)

slice_slider.observe(
    draw_selected_slice,
    names="value",
)

denoised_checkbox.observe(
    draw_selected_slice,
    names="value",
)

update_slice_range()
draw_selected_slice()

display(
    widgets.VBox(
        [
            phase_slider,
            slice_slider,
            denoised_checkbox,
            browser_output,
        ]
    )
)

## 7. Select the canonical phase and create the MedGS image dataset

All slices from the selected phase are used. Each HU slice is windowed to `[0, 1]`, converted to RGB PNG, and written both in its original orientation and as a horizontal mirror, matching the existing MedGS input convention.

In [20]:
matching_phases = [
    phase
    for phase in available_phases
    if np.isclose(phase, CANONICAL_PHASE_PERCENT)
]
if len(matching_phases) != 1:
    raise ValueError(
        f"Canonical phase {CANONICAL_PHASE_PERCENT:g}% is unavailable. "
        f"Available phases: {available_phases}"
    )

canonical_phase = float(matching_phases[0])
canonical_table = phase_slice_tables[canonical_phase]
representation_name = "denoised" if TRAIN_ON_DENOISED else "raw"

run_name = (
    f"patient_{PATIENT_ID}"
    f"__phase_{phase_tag(canonical_phase)}"
    f"__{representation_name}"
    f"__poly{POLY_DEGREE}"
    f"__iter{ITERATIONS}"
)

run_root = CANONICAL_EXPERIMENTS_ROOT / run_name
medgs_dataset_root = run_root / "dataset"
original_images_root = medgs_dataset_root / "original"
mirror_images_root = medgs_dataset_root / "mirror"
model_root = run_root / "model"

source_volume_path = phase_volume_paths[
    (canonical_phase, TRAIN_ON_DENOISED)
]
source_volume = np.load(source_volume_path, mmap_mode="r")

existing_originals = sorted(original_images_root.glob("*.png"))
expected_frames = len(canonical_table)
can_reuse_dataset = (
        not REBUILD_MEDGS_DATASET
        and len(existing_originals) == expected_frames
        and len(list(mirror_images_root.glob("*.png"))) == expected_frames
)

if can_reuse_dataset:
    print(f"Reusing existing MedGS dataset: {medgs_dataset_root}")
else:
    shutil.rmtree(medgs_dataset_root, ignore_errors=True)
    original_images_root.mkdir(parents=True)
    mirror_images_root.mkdir(parents=True)

    frame_rows = []
    for frame_index in range(expected_frames):
        normalized = window_hu(
            np.asarray(source_volume[frame_index]),
            HU_WINDOW_LOW,
            HU_WINDOW_HIGH,
        )
        rgb = grayscale_to_rgb_uint8(normalized)
        filename = f"{frame_index:04d}.png"
        original_path = original_images_root / filename
        mirror_path = mirror_images_root / filename
        Image.fromarray(rgb).save(original_path)
        Image.fromarray(np.fliplr(rgb)).save(mirror_path)

        row = canonical_table.iloc[frame_index]
        frame_rows.append(
            {
                "FrameIndex": frame_index,
                "SliceIndex": int(row["SliceIndex"]),
                "SliceCoordinate": float(row["SliceCoordinate"]),
                "DICOMPath": str(row["Path"]),
                "OriginalPNG": str(original_path),
                "MirrorPNG": str(mirror_path),
            }
        )

    pd.DataFrame(frame_rows).to_csv(
        run_root / "frame_manifest.csv", index=False
    )

print(f"Canonical phase:       {canonical_phase:g}%")
print(f"Input representation:  {representation_name}")
print(f"Canonical slices:      {expected_frames}")
print(f"MedGS dataset:         {medgs_dataset_root}")
print(f"Model output:          {model_root}")

Canonical phase:       20%
Input representation:  raw
Canonical slices:      147
MedGS dataset:         /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/dataset
Model output:          /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/model


## 8. Train the canonical MedGS model

The command is the same image pipeline used previously: quadratic folding, mirror cameras, batch size 3, and a checkpoint at the final iteration. A complete existing model is reused when `REUSE_COMPLETED_MODEL=True`.

### Train original (not denoised)

In [23]:
train_script = MEDGS_REPOSITORY / "train.py"

completed_model_marker = (
        model_root
        / "point_cloud"
        / f"iteration_{ITERATIONS}"
        / "point_cloud.ply"
)

train_command = [
    sys.executable,
    str(train_script),
    "-s",
    str(medgs_dataset_root),
    "-m",
    str(model_root),
    "--pipeline",
    "img",
    "--iterations",
    str(ITERATIONS),
    "--poly_degree",
    str(POLY_DEGREE),
    "--batch_size",
    str(BATCH_SIZE),
    "--camera",
    CAMERA,
    "--test_iterations",
    str(ITERATIONS),
    "--save_iterations",
    str(ITERATIONS),
    "--checkpoint_iterations",
    str(ITERATIONS),
]

run_config = {
    "patient_id": PATIENT_ID,
    "study_instance_uid": STUDY_INSTANCE_UID,
    "canonical_phase_percent": canonical_phase,
    "input_representation": representation_name,
    "source_volume": str(source_volume_path),
    "hu_window": [
        HU_WINDOW_LOW,
        HU_WINDOW_HIGH,
    ],
    "denoise_sigma_voxels": list(DENOISE_SIGMA),
    "medgs_dataset": str(medgs_dataset_root),
    "model_root": str(model_root),
    "iterations": ITERATIONS,
    "poly_degree": POLY_DEGREE,
    "batch_size": BATCH_SIZE,
    "camera": CAMERA,
    "command": train_command,
}

run_root.mkdir(
    parents=True,
    exist_ok=True,
)

(run_root / "run.json").write_text(
    json.dumps(run_config, indent=2) + "\n",
    encoding="utf-8",
)

print("Training command:")
print(shlex.join(train_command))

if completed_model_marker.is_file() and REUSE_COMPLETED_MODEL:
    print(f"Complete model already exists; reusing {model_root}")

elif RUN_TRAINING:
    shutil.rmtree(
        model_root,
        ignore_errors=True,
    )

    subprocess.run(
        train_command,
        cwd=MEDGS_REPOSITORY,
        check=True,
    )

else:
    print("Training is disabled by RUN_TRAINING=False.")

print(f"Expected final model marker: {completed_model_marker}")
print(f"Marker exists:               {completed_model_marker.is_file()}")

Training command:
/home/jovyan/shared/mtm_medgs_stack/envs/medgs-worf/bin/python /home/jovyan/shared/mtm_medgs_stack/repo/MedGS/train.py -s /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/dataset -m /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/model --pipeline img --iterations 30000 --poly_degree 2 --batch_size 3 --camera mirror --test_iterations 30000 --save_iterations 30000 --checkpoint_iterations 30000
torch cuda:  True
Optimizing /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/model
Training with photometric loss [29/07 12:47:43]
Output folder: /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/model [29/07 12:47:43]
Tensorboard not available: not logging progress [29/07 12:47:43]
frames 147 [29/07 12:47:43]
Distance: 1.0 [29/

Training progress:  53%|█████▎    | 15900/30000 [17:12<23:08, 10.15it/s, Loss=0.0369916, psnr=36.71, point=1296510]

prev_next_overlap 2 [29/07 12:47:53]

[ITER 600] Densifying Gaussians [29/07 12:48:12]

[ITER 700] Densifying Gaussians [29/07 12:48:15]

[ITER 800] Densifying Gaussians [29/07 12:48:19]

[ITER 900] Densifying Gaussians [29/07 12:48:22]

[ITER 1000] Densifying Gaussians [29/07 12:48:26]

[ITER 1100] Densifying Gaussians [29/07 12:48:29]

[ITER 1200] Densifying Gaussians [29/07 12:48:32]

[ITER 1300] Densifying Gaussians [29/07 12:48:36]

[ITER 1400] Densifying Gaussians [29/07 12:48:39]

[ITER 1500] Densifying Gaussians [29/07 12:48:43]

[ITER 1600] Densifying Gaussians [29/07 12:48:46]

[ITER 1700] Densifying Gaussians [29/07 12:48:50]

[ITER 1800] Densifying Gaussians [29/07 12:48:54]

[ITER 1900] Densifying Gaussians [29/07 12:48:58]

[ITER 2000] Densifying Gaussians [29/07 12:49:02]

[ITER 2100] Densifying Gaussians [29/07 12:49:06]

[ITER 2200] Densifying Gaussians [29/07 12:49:10]

[ITER 2300] Densifying Gaussians [29/07 12:49:14]

[ITER 2400] Densifying Gaussians [29/07 12:49:18

Training progress: 100%|██████████| 30000/30000 [39:56<00:00, 12.52it/s, Loss=0.0365584, psnr=36.14, point=1410016]



[ITER 15900] Densifying Gaussians [29/07 13:05:06]

[ITER 16000] Densifying Gaussians [29/07 13:05:16]

[ITER 16100] Densifying Gaussians [29/07 13:05:26]

[ITER 16200] Densifying Gaussians [29/07 13:05:36]

[ITER 16300] Densifying Gaussians [29/07 13:05:46]

[ITER 16400] Densifying Gaussians [29/07 13:05:56]

[ITER 16500] Densifying Gaussians [29/07 13:06:06]

[ITER 16600] Densifying Gaussians [29/07 13:06:16]

[ITER 16700] Densifying Gaussians [29/07 13:06:26]

[ITER 16800] Densifying Gaussians [29/07 13:06:36]

[ITER 16900] Densifying Gaussians [29/07 13:06:46]

[ITER 17000] Densifying Gaussians [29/07 13:06:56]

[ITER 17100] Densifying Gaussians [29/07 13:07:06]

[ITER 17200] Densifying Gaussians [29/07 13:07:16]

[ITER 17300] Densifying Gaussians [29/07 13:07:26]

[ITER 17400] Densifying Gaussians [29/07 13:07:36]

[ITER 17500] Densifying Gaussians [29/07 13:07:46]

[ITER 17600] Densifying Gaussians [29/07 13:07:57]

[ITER 17700] Densifying Gaussians [29/07 13:08:07]

[ITER 17800

In [25]:
from skimage.metrics import (
    peak_signal_noise_ratio,
    structural_similarity,
)
import lpips


def load_gray_image(path: Path) -> np.ndarray:
    """Load a PNG as a normalized grayscale array."""

    with Image.open(path) as image:
        return (
                np.asarray(
                    image.convert("L"),
                    dtype=np.float32,
                )
                / 255.0
        )


def lpips_distance(
        model: torch.nn.Module,
        reference: np.ndarray,
        reconstruction: np.ndarray,
) -> float:
    """Compute LPIPS for two normalized grayscale images."""

    reference_tensor = (
            torch.from_numpy(
                np.ascontiguousarray(reference)
            )
            .float()
            .unsqueeze(0)
            .unsqueeze(0)
            .repeat(1, 3, 1, 1)
            .cuda()
            * 2.0
            - 1.0
    )

    reconstruction_tensor = (
            torch.from_numpy(
                np.ascontiguousarray(reconstruction)
            )
            .float()
            .unsqueeze(0)
            .unsqueeze(0)
            .repeat(1, 3, 1, 1)
            .cuda()
            * 2.0
            - 1.0
    )

    with torch.no_grad():
        return float(
            model(
                reference_tensor,
                reconstruction_tensor,
            ).item()
        )


def render_and_measure_current_model():
    """Render and evaluate the model selected by current run variables."""

    render_root = model_root / "render_img"

    shutil.rmtree(
        render_root,
        ignore_errors=True,
    )

    render_command = [
        sys.executable,
        str(MEDGS_REPOSITORY / "render.py"),
        "-s",
        str(medgs_dataset_root),
        "-m",
        str(model_root),
        "--iteration",
        str(ITERATIONS),
        "--poly_degree",
        str(POLY_DEGREE),
        "--camera",
        CAMERA,
        "--pipeline",
        "img",
        "--interp",
        "1",
    ]

    print("Rendering final model:")
    print(shlex.join(render_command))

    subprocess.run(
        render_command,
        cwd=MEDGS_REPOSITORY,
        check=True,
    )

    current_frame_manifest_df = pd.read_csv(
        run_root / "frame_manifest.csv"
    )

    current_lpips_model = (
        lpips.LPIPS(net="alex")
        .cuda()
        .eval()
    )

    metric_rows = []

    for _, frame in current_frame_manifest_df.iterrows():
        frame_index = int(frame["FrameIndex"])

        original_path = Path(frame["OriginalPNG"])
        reconstruction_path = (
                render_root
                / f"{frame_index:05d}_0.png"
        )

        original = load_gray_image(original_path)
        reconstruction = load_gray_image(
            reconstruction_path
        )

        metric_rows.append(
            {
                "FrameIndex": frame_index,
                "SliceIndex": int(frame["SliceIndex"]),
                "SliceCoordinate": float(
                    frame["SliceCoordinate"]
                ),
                "PSNR": peak_signal_noise_ratio(
                    original,
                    reconstruction,
                    data_range=1.0,
                ),
                "SSIM": structural_similarity(
                    original,
                    reconstruction,
                    data_range=1.0,
                ),
                "LPIPS": lpips_distance(
                    current_lpips_model,
                    original,
                    reconstruction,
                ),
                "OriginalPath": str(original_path),
                "ReconstructionPath": str(
                    reconstruction_path
                ),
            }
        )

    current_metrics_df = pd.DataFrame(metric_rows)

    current_summary_df = pd.DataFrame(
        [
            {
                "PatientID": PATIENT_ID,
                "StudyInstanceUID": STUDY_INSTANCE_UID,
                "PhasePercent": canonical_phase,
                "Representation": representation_name,
                "Slices": len(current_metrics_df),
                "PSNR": current_metrics_df["PSNR"].mean(),
                "SSIM": current_metrics_df["SSIM"].mean(),
                "LPIPS": current_metrics_df["LPIPS"].mean(),
            }
        ]
    )

    current_metrics_df.to_csv(
        run_root / "train_metrics.csv",
        index=False,
    )

    current_summary_df.to_csv(
        run_root / "train_metrics_summary.csv",
        index=False,
    )

    del current_lpips_model
    torch.cuda.empty_cache()

    return (
        current_frame_manifest_df,
        current_metrics_df,
        current_summary_df,
    )


(
    frame_manifest_df,
    train_metrics_df,
    train_metrics_summary_df,
) = render_and_measure_current_model()

print(f"Metrics saved to: {run_root / 'train_metrics.csv'}")
display(train_metrics_summary_df)
# display(train_metrics_df)

Rendering final model:
/home/jovyan/shared/mtm_medgs_stack/envs/medgs-worf/bin/python /home/jovyan/shared/mtm_medgs_stack/repo/MedGS/render.py -s /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/dataset -m /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/model --iteration 30000 --poly_degree 2 --camera mirror --pipeline img --interp 1


/home/jovyan/shared/mtm_medgs_stack/repo/MedGS/render.py:168: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_state, loaded_iter = torch.load(ckpt_path, map_location="cu

Looking for config file in /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/model/cfg_args
Config file found: /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/model/cfg_args
Rendering /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/model
Distance: 1.0 [29/07 13:28:51]
Loading trained model at iteration 30000 [29/07 13:28:51]
Creating Training Transforms [29/07 13:28:51]
Creating Test Transforms [29/07 13:28:56]
AAAAA radius 1.1 [29/07 13:28:59]
AAAAA translate [-0. -0. -0.] center [0. 0. 0.] [29/07 13:28:59]
Generating random point cloud (100000)... [29/07 13:28:59]
Loading Training Cameras [29/07 13:28:59]
Loading Test Cameras [29/07 13:29:00]
Loading checkpoint: /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/model/chkpnt30000.pth

Rendering progress: 100%|██████████| 147/147 [00:06<00:00, 21.74it/s]


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/home/jovyan/shared/mtm_medgs_stack/envs/medgs-worf/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/jovyan/shared/mtm_medgs_stack/envs/medgs-worf/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /home/jovyan/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth
100%|██████████| 233M/233M [00:02<00:00, 106MB/s]  
/home/jovyan/shared/mtm_medgs_stack/envs/medgs-worf/lib/python3.11/site-packages/lpips/lpips.py:107

Loading model from: /home/jovyan/shared/mtm_medgs_stack/envs/medgs-worf/lib/python3.11/site-packages/lpips/weights/v0.1/alex.pth
Metrics saved to: /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/train_metrics.csv


,PatientID,StudyInstanceUID,PhasePercent,Representation,Slices,PSNR,SSIM,LPIPS
0,117_HM10395,1.3.6.1.4.1.14519.5.2.1.6834.5010.378204929111417980831212264180,20.0,raw,147,37.876642,0.938719,0.086408


,FrameIndex,SliceIndex,SliceCoordinate,PSNR,SSIM,LPIPS,OriginalPath,ReconstructionPath
0,0,0,-266.2,38.687058,0.943455,0.072683,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/dat...,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/mod...
1,1,1,-263.2,40.323765,0.953662,0.073976,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/dat...,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/mod...
2,2,2,-260.2,39.550590,0.946209,0.080697,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/dat...,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/mod...
3,3,3,-257.2,39.728602,0.949805,0.077676,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/dat...,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/mod...
4,4,4,-254.2,38.702013,0.933900,0.092153,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/dat...,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/mod...
5,5,5,-251.2,39.717573,0.951096,0.073987,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/dat...,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/mod...
6,6,6,-248.2,39.645944,0.950293,0.075053,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/dat...,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/mod...
7,7,7,-245.2,39.626318,0.949787,0.073330,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/dat...,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/mod...
8,8,8,-242.2,39.739775,0.951177,0.070046,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/dat...,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/mod...
9,9,9,-239.2,39.286302,0.944613,0.079988,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/dat...,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__phase_20__raw__poly2__iter30000/mod...


In [26]:
print(
    "Creating the canonical input/reconstruction browser."
)

comparison_slider = widgets.IntSlider(
    value=len(frame_manifest_df) // 2,
    min=0,
    max=len(frame_manifest_df) - 1,
    step=1,
    description="Slice",
    continuous_update=False,
    layout=widgets.Layout(width="700px"),
)

comparison_output = widgets.Output()


def draw_canonical_comparison(
        change: dict | None = None,
) -> None:
    """Display a fixed-size input/reconstruction canvas."""

    position = int(comparison_slider.value)
    frame = frame_manifest_df.iloc[position]

    frame_index = int(frame["FrameIndex"])
    slice_index = int(frame["SliceIndex"])
    coordinate = float(frame["SliceCoordinate"])

    original = load_gray_image(
        Path(frame["OriginalPNG"])
    )

    reconstruction_path = (
            model_root
            / "render_img"
            / f"{frame_index:05d}_0.png"
    )

    if reconstruction_path.is_file():
        reconstruction = load_gray_image(
            reconstruction_path
        )

        metric_row = train_metrics_df.loc[
            train_metrics_df["FrameIndex"]
            == frame_index
            ].iloc[0]

        metric_text = (
            f"PSNR {metric_row['PSNR']:.3f} dB | "
            f"SSIM {metric_row['SSIM']:.4f} | "
            f"LPIPS {metric_row['LPIPS']:.4f}"
        )
    else:
        reconstruction = np.zeros_like(original)
        metric_text = "Reconstruction unavailable"

    separator = np.ones(
        (original.shape[0], 8),
        dtype=np.float32,
    )

    comparison_canvas = np.concatenate(
        [
            original,
            separator,
            reconstruction,
        ],
        axis=1,
    )

    with comparison_output:
        clear_output(wait=True)

        plt.figure(figsize=(14, 7))
        plt.imshow(
            comparison_canvas,
            cmap="gray",
            vmin=0.0,
            vmax=1.0,
        )

        plt.title(
            f"Original {representation_name} input"
            f"                         "
            f"MedGS reconstruction\n"
            f"Patient {PATIENT_ID} | "
            f"phase {canonical_phase:g}% | "
            f"slice {slice_index} | "
            f"coordinate {coordinate:.3f} mm\n"
            f"{metric_text}"
        )

        plt.axis("off")
        plt.tight_layout()
        plt.show()


comparison_slider.observe(
    draw_canonical_comparison,
    names="value",
)

draw_canonical_comparison()

display(
    widgets.VBox(
        [
            comparison_slider,
            comparison_output,
        ]
    )
)

Creating the canonical input/reconstruction browser.


### Train denoised

In [28]:
TRAIN_ON_DENOISED = True

canonical_phase = float(CANONICAL_PHASE_PERCENT)
study_tag = str(STUDY_INSTANCE_UID)[-12:]
representation_name = "denoised"

patient_prepared_root = (
        PREPARED_4D_ROOT
        / PATIENT_ID
        / f"study_{study_tag}"
)

source_volume_path = (
        patient_prepared_root
        / "volumes"
        / "denoised"
        / f"phase_{phase_tag(canonical_phase)}.npy"
)

source_volume = np.load(
    source_volume_path,
    mmap_mode="r",
)

canonical_table = (
    phase_slice_tables[canonical_phase]
    .reset_index(drop=True)
)

run_name = (
    f"patient_{PATIENT_ID}"
    f"__study_{study_tag}"
    f"__phase_{phase_tag(canonical_phase)}"
    f"__{representation_name}"
    f"__poly{POLY_DEGREE}"
    f"__iter{ITERATIONS}"
)

run_root = CANONICAL_EXPERIMENTS_ROOT / run_name
medgs_dataset_root = run_root / "dataset"
model_root = run_root / "model"

original_images_root = medgs_dataset_root / "original"
mirror_images_root = medgs_dataset_root / "mirror"
frame_manifest_path = run_root / "frame_manifest.csv"

if REBUILD_MEDGS_DATASET or not frame_manifest_path.is_file():
    shutil.rmtree(
        medgs_dataset_root,
        ignore_errors=True,
    )

    original_images_root.mkdir(
        parents=True,
    )

    mirror_images_root.mkdir(
        parents=True,
    )

    frame_rows = []

    for frame_index in range(len(canonical_table)):
        normalized = window_hu(
            np.asarray(source_volume[frame_index]),
            HU_WINDOW_LOW,
            HU_WINDOW_HIGH,
        )

        rgb = grayscale_to_rgb_uint8(normalized)
        filename = f"{frame_index:04d}.png"

        original_path = (
                original_images_root
                / filename
        )

        mirror_path = (
                mirror_images_root
                / filename
        )

        Image.fromarray(rgb).save(original_path)

        Image.fromarray(
            np.fliplr(rgb).copy()
        ).save(mirror_path)

        row = canonical_table.iloc[frame_index]

        frame_rows.append(
            {
                "PatientID": PATIENT_ID,
                "StudyInstanceUID": STUDY_INSTANCE_UID,
                "PhasePercent": canonical_phase,
                "FrameIndex": frame_index,
                "SliceIndex": int(row["SliceIndex"]),
                "SliceCoordinate": float(
                    row["SliceCoordinate"]
                ),
                "DICOMPath": str(row["Path"]),
                "SourceVolumePath": str(
                    source_volume_path
                ),
                "OriginalPNG": str(original_path),
                "MirrorPNG": str(mirror_path),
            }
        )

    frame_manifest_df = pd.DataFrame(frame_rows)

    frame_manifest_df.to_csv(
        frame_manifest_path,
        index=False,
    )

else:
    frame_manifest_df = pd.read_csv(
        frame_manifest_path
    )

completed_model_marker = (
        model_root
        / "point_cloud"
        / f"iteration_{ITERATIONS}"
        / "point_cloud.ply"
)

train_command = [
    sys.executable,
    str(MEDGS_REPOSITORY / "train.py"),
    "-s",
    str(medgs_dataset_root),
    "-m",
    str(model_root),
    "--pipeline",
    "img",
    "--iterations",
    str(ITERATIONS),
    "--poly_degree",
    str(POLY_DEGREE),
    "--batch_size",
    str(BATCH_SIZE),
    "--camera",
    CAMERA,
    "--test_iterations",
    str(ITERATIONS),
    "--save_iterations",
    str(ITERATIONS),
    "--checkpoint_iterations",
    str(ITERATIONS),
]

print(f"Patient:               {PATIENT_ID}")
print(f"Study:                 {STUDY_INSTANCE_UID}")
print(f"Canonical phase:       {canonical_phase:g}%")
print(f"Input representation:  {representation_name}")
print(f"Source volume:         {source_volume_path}")
print(f"Canonical slices:      {len(canonical_table)}")
print(f"MedGS dataset:         {medgs_dataset_root}")
print(f"Model output:          {model_root}")
print()
print("Training command:")
print(shlex.join(train_command))

if completed_model_marker.is_file() and REUSE_COMPLETED_MODEL:
    print("Reusing completed denoised model.")
else:
    shutil.rmtree(
        model_root,
        ignore_errors=True,
    )

    subprocess.run(
        train_command,
        cwd=MEDGS_REPOSITORY,
        check=True,
    )

Patient:               117_HM10395
Study:                 1.3.6.1.4.1.14519.5.2.1.6834.5010.378204929111417980831212264180
Canonical phase:       20%
Input representation:  denoised
Source volume:         /home/jovyan/shared/mtm_medgs_stack/results/prepared_4d_patients/117_HM10395/study_831212264180/volumes/denoised/phase_20.npy
Canonical slices:      147
MedGS dataset:         /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoised__poly2__iter30000/dataset
Model output:          /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoised__poly2__iter30000/model

Training command:
/home/jovyan/shared/mtm_medgs_stack/envs/medgs-worf/bin/python /home/jovyan/shared/mtm_medgs_stack/repo/MedGS/train.py -s /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoised__poly2__iter30000/dataset -m /home/jovyan/sha

Training progress:  53%|█████▎    | 15900/30000 [15:08<17:19, 13.56it/s, Loss=0.0241909, psnr=39.33, point=604595]

prev_next_overlap 2 [29/07 13:35:28]

[ITER 600] Densifying Gaussians [29/07 13:36:22]

[ITER 700] Densifying Gaussians [29/07 13:36:26]

[ITER 800] Densifying Gaussians [29/07 13:36:29]

[ITER 900] Densifying Gaussians [29/07 13:36:32]

[ITER 1000] Densifying Gaussians [29/07 13:36:36]

[ITER 1100] Densifying Gaussians [29/07 13:36:39]

[ITER 1200] Densifying Gaussians [29/07 13:36:42]

[ITER 1300] Densifying Gaussians [29/07 13:36:45]

[ITER 1400] Densifying Gaussians [29/07 13:36:49]

[ITER 1500] Densifying Gaussians [29/07 13:36:52]

[ITER 1600] Densifying Gaussians [29/07 13:36:55]

[ITER 1700] Densifying Gaussians [29/07 13:36:59]

[ITER 1800] Densifying Gaussians [29/07 13:37:02]

[ITER 1900] Densifying Gaussians [29/07 13:37:05]

[ITER 2000] Densifying Gaussians [29/07 13:37:09]

[ITER 2100] Densifying Gaussians [29/07 13:37:13]

[ITER 2200] Densifying Gaussians [29/07 13:37:17]

[ITER 2300] Densifying Gaussians [29/07 13:37:21]

[ITER 2400] Densifying Gaussians [29/07 13:37:24

Training progress: 100%|██████████| 30000/30000 [31:53<00:00, 15.68it/s, Loss=0.0240901, psnr=39.22, point=653206]



[ITER 15900] Densifying Gaussians [29/07 13:50:35]

[ITER 16000] Densifying Gaussians [29/07 13:50:43]

[ITER 16100] Densifying Gaussians [29/07 13:50:50]

[ITER 16200] Densifying Gaussians [29/07 13:50:57]

[ITER 16300] Densifying Gaussians [29/07 13:51:05]

[ITER 16400] Densifying Gaussians [29/07 13:51:12]

[ITER 16500] Densifying Gaussians [29/07 13:51:20]

[ITER 16600] Densifying Gaussians [29/07 13:51:27]

[ITER 16700] Densifying Gaussians [29/07 13:51:34]

[ITER 16800] Densifying Gaussians [29/07 13:51:42]

[ITER 16900] Densifying Gaussians [29/07 13:51:49]

[ITER 17000] Densifying Gaussians [29/07 13:51:57]

[ITER 17100] Densifying Gaussians [29/07 13:52:04]

[ITER 17200] Densifying Gaussians [29/07 13:52:12]

[ITER 17300] Densifying Gaussians [29/07 13:52:19]

[ITER 17400] Densifying Gaussians [29/07 13:52:27]

[ITER 17500] Densifying Gaussians [29/07 13:52:34]

[ITER 17600] Densifying Gaussians [29/07 13:52:42]

[ITER 17700] Densifying Gaussians [29/07 13:52:50]

[ITER 17800

In [33]:
(
    frame_manifest_df,
    denoised_target_metrics_df,
    denoised_target_summary_df,
) = render_and_measure_current_model()

raw_volume_path = (
    PREPARED_4D_ROOT
    / PATIENT_ID
    / f"study_{str(STUDY_INSTANCE_UID)[-12:]}"
    / "volumes"
    / "raw"
    / f"phase_{phase_tag(canonical_phase)}.npy"
)

raw_volume = np.load(
    raw_volume_path,
    mmap_mode="r",
)

raw_lpips_model = (
    lpips.LPIPS(net="alex")
    .cuda()
    .eval()
)

raw_metric_rows = []

for _, frame in frame_manifest_df.iterrows():
    frame_index = int(frame["FrameIndex"])
    slice_index = int(frame["SliceIndex"])

    reconstruction_path = (
        model_root
        / "render_img"
        / f"{frame_index:05d}_0.png"
    )

    reconstruction = load_gray_image(
        reconstruction_path
    )

    raw_target = window_hu(
        np.asarray(raw_volume[slice_index]),
        HU_WINDOW_LOW,
        HU_WINDOW_HIGH,
    )

    raw_target = (
        grayscale_to_rgb_uint8(raw_target)[:, :, 0]
        .astype(np.float32)
        / 255.0
    )

    raw_metric_rows.append(
        {
            "FrameIndex": frame_index,
            "PSNR_vs_raw": peak_signal_noise_ratio(
                raw_target,
                reconstruction,
                data_range=1.0,
            ),
            "SSIM_vs_raw": structural_similarity(
                raw_target,
                reconstruction,
                data_range=1.0,
            ),
            "LPIPS_vs_raw": lpips_distance(
                raw_lpips_model,
                raw_target,
                reconstruction,
            ),
        }
    )

raw_metrics_df = pd.DataFrame(raw_metric_rows)

denoised_metrics_df = (
    denoised_target_metrics_df.rename(
        columns={
            "PSNR": "PSNR_vs_denoised",
            "SSIM": "SSIM_vs_denoised",
            "LPIPS": "LPIPS_vs_denoised",
        }
    )
    .merge(
        raw_metrics_df,
        on="FrameIndex",
    )
)

denoised_metrics_summary_df = pd.DataFrame(
    [
        {
            "PatientID": PATIENT_ID,
            "StudyInstanceUID": STUDY_INSTANCE_UID,
            "PhasePercent": canonical_phase,
            "ModelInput": "denoised",
            "EvaluationTarget": "denoised",
            "Slices": len(denoised_metrics_df),
            "PSNR": denoised_metrics_df[
                "PSNR_vs_denoised"
            ].mean(),
            "SSIM": denoised_metrics_df[
                "SSIM_vs_denoised"
            ].mean(),
            "LPIPS": denoised_metrics_df[
                "LPIPS_vs_denoised"
            ].mean(),
        },
        {
            "PatientID": PATIENT_ID,
            "StudyInstanceUID": STUDY_INSTANCE_UID,
            "PhasePercent": canonical_phase,
            "ModelInput": "denoised",
            "EvaluationTarget": "raw",
            "Slices": len(denoised_metrics_df),
            "PSNR": denoised_metrics_df[
                "PSNR_vs_raw"
            ].mean(),
            "SSIM": denoised_metrics_df[
                "SSIM_vs_raw"
            ].mean(),
            "LPIPS": denoised_metrics_df[
                "LPIPS_vs_raw"
            ].mean(),
        },
    ]
)

denoised_metrics_path = (
    run_root
    / "denoised_metrics.csv"
)

denoised_summary_path = (
    run_root
    / "denoised_metrics_summary.csv"
)

denoised_metrics_df.to_csv(
    denoised_metrics_path,
    index=False,
)

denoised_metrics_summary_df.to_csv(
    denoised_summary_path,
    index=False,
)

print(f"Metrics saved to: {denoised_metrics_path}")
print(f"Summary saved to: {denoised_summary_path}")

display(denoised_metrics_summary_df)
display(denoised_metrics_df)

Rendering final model:
/home/jovyan/shared/mtm_medgs_stack/envs/medgs-worf/bin/python /home/jovyan/shared/mtm_medgs_stack/repo/MedGS/render.py -s /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoised__poly2__iter30000/dataset -m /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoised__poly2__iter30000/model --iteration 30000 --poly_degree 2 --camera mirror --pipeline img --interp 1


/home/jovyan/shared/mtm_medgs_stack/repo/MedGS/render.py:168: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_state, loaded_iter = torch.load(ckpt_path, map_location="cu

Looking for config file in /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoised__poly2__iter30000/model/cfg_args
Config file found: /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoised__poly2__iter30000/model/cfg_args
Rendering /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoised__poly2__iter30000/model
Distance: 1.0 [29/07 14:11:12]
Loading trained model at iteration 30000 [29/07 14:11:12]
Creating Training Transforms [29/07 14:11:12]
Creating Test Transforms [29/07 14:11:17]
AAAAA radius 1.1 [29/07 14:11:20]
AAAAA translate [-0. -0. -0.] center [0. 0. 0.] [29/07 14:11:20]
Generating random point cloud (100000)... [29/07 14:11:20]
Loading Training Cameras [29/07 14:11:20]
Loading Test Cameras [29/07 14:11:21]
Loading checkpoint: /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs

Rendering progress: 100%|██████████| 147/147 [00:05<00:00, 25.69it/s]


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


...

Loading model from: /home/jovyan/shared/mtm_medgs_stack/envs/medgs-worf/lib/python3.11/site-packages/lpips/weights/v0.1/alex.pth


/home/jovyan/shared/mtm_medgs_stack/envs/medgs-worf/lib/python3.11/site-packages/lpips/lpips.py:107: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch

Metrics saved to: /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoised__poly2__iter30000/denoised_metrics.csv
Summary saved to: /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoised__poly2__iter30000/denoised_metrics_summary.csv


,PatientID,StudyInstanceUID,PhasePercent,ModelInput,EvaluationTarget,Slices,PSNR,SSIM,LPIPS
0,117_HM10395,1.3.6.1.4.1.14519.5.2.1.6834.5010.378204929111417980831212264180,20.0,denoised,denoised,147,40.621407,0.962294,0.057387
1,117_HM10395,1.3.6.1.4.1.14519.5.2.1.6834.5010.378204929111417980831212264180,20.0,denoised,raw,147,33.486089,0.886973,0.177803


,FrameIndex,SliceIndex,SliceCoordinate,PSNR_vs_denoised,SSIM_vs_denoised,LPIPS_vs_denoised,OriginalPath,ReconstructionPath,PSNR_vs_raw,SSIM_vs_raw,LPIPS_vs_raw
0,0,0,-266.2,41.625067,0.964838,0.044998,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoi...,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoi...,34.334724,0.898506,0.158561
1,1,1,-263.2,43.410461,0.971483,0.047173,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoi...,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoi...,34.766669,0.904760,0.164757
2,2,2,-260.2,42.729618,0.967713,0.051926,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoi...,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoi...,34.604528,0.896885,0.178482
3,3,3,-257.2,42.849514,0.969272,0.050161,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoi...,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoi...,34.721770,0.901676,0.175032
4,4,4,-254.2,41.807485,0.962916,0.057030,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoi...,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoi...,34.545322,0.897559,0.177149
5,5,5,-251.2,42.796925,0.970089,0.047476,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoi...,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoi...,34.863488,0.908326,0.169323
6,6,6,-248.2,42.619859,0.968516,0.047842,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoi...,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoi...,34.836053,0.905630,0.167943
7,7,7,-245.2,42.567058,0.968354,0.046548,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoi...,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoi...,34.892810,0.906381,0.164926
8,8,8,-242.2,42.613329,0.968838,0.044952,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoi...,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoi...,34.950736,0.907366,0.161947
9,9,9,-239.2,42.274292,0.967358,0.048220,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoi...,/home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoi...,34.847793,0.903402,0.165398


In [32]:
print(
    "Creating the raw input / denoised-model reconstruction browser."
)

comparison_slider = widgets.IntSlider(
    value=len(frame_manifest_df) // 2,
    min=0,
    max=len(frame_manifest_df) - 1,
    step=1,
    description="Slice",
    continuous_update=False,
    layout=widgets.Layout(width="700px"),
)

comparison_output = widgets.Output()


def draw_denoised_model_comparison(
    change: dict | None = None,
) -> None:
    position = int(comparison_slider.value)
    frame = frame_manifest_df.iloc[position]

    frame_index = int(frame["FrameIndex"])
    slice_index = int(frame["SliceIndex"])
    coordinate = float(frame["SliceCoordinate"])

    raw_target = window_hu(
        np.asarray(raw_volume[slice_index]),
        HU_WINDOW_LOW,
        HU_WINDOW_HIGH,
    )

    raw_target = (
        grayscale_to_rgb_uint8(raw_target)[:, :, 0]
        .astype(np.float32)
        / 255.0
    )

    reconstruction_path = (
        model_root
        / "render_img"
        / f"{frame_index:05d}_0.png"
    )

    if reconstruction_path.is_file():
        reconstruction = load_gray_image(
            reconstruction_path
        )

        metric_row = denoised_metrics_df.loc[
            denoised_metrics_df["FrameIndex"]
            == frame_index
        ].iloc[0]

        metric_text = (
            f"PSNR {metric_row['PSNR_vs_raw']:.3f} dB | "
            f"SSIM {metric_row['SSIM_vs_raw']:.4f} | "
            f"LPIPS {metric_row['LPIPS_vs_raw']:.4f}"
        )
    else:
        reconstruction = np.zeros_like(raw_target)
        metric_text = "Reconstruction unavailable"

    separator = np.ones(
        (raw_target.shape[0], 8),
        dtype=np.float32,
    )

    comparison_canvas = np.concatenate(
        [
            raw_target,
            separator,
            reconstruction,
        ],
        axis=1,
    )

    with comparison_output:
        clear_output(wait=True)

        plt.figure(figsize=(14, 7))
        plt.imshow(
            comparison_canvas,
            cmap="gray",
            vmin=0.0,
            vmax=1.0,
        )

        plt.title(
            "Original raw CT"
            "                         "
            "MedGS reconstruction trained on denoised CT\n"
            f"Patient {PATIENT_ID} | "
            f"phase {canonical_phase:g}% | "
            f"slice {slice_index} | "
            f"coordinate {coordinate:.3f} mm\n"
            f"{metric_text}"
        )

        plt.axis("off")
        plt.tight_layout()
        plt.show()


comparison_slider.observe(
    draw_denoised_model_comparison,
    names="value",
)

draw_denoised_model_comparison()

display(
    widgets.VBox(
        [
            comparison_slider,
            comparison_output,
        ]
    )
)

Creating the raw input / denoised-model reconstruction browser.


In [35]:
from skimage.metrics import (
    peak_signal_noise_ratio,
    structural_similarity,
)
import lpips


def load_gray_image(path: Path) -> np.ndarray:
    """Load a PNG as a normalized grayscale array."""

    with Image.open(path) as image:
        return (
            np.asarray(
                image.convert("L"),
                dtype=np.float32,
            )
            / 255.0
        )


def volume_slice_to_gray(
    volume: np.ndarray,
    slice_index: int,
) -> np.ndarray:
    """Convert one HU slice to the same normalized grayscale representation."""

    windowed_slice = window_hu(
        np.asarray(volume[slice_index]),
        HU_WINDOW_LOW,
        HU_WINDOW_HIGH,
    )

    return (
        grayscale_to_rgb_uint8(windowed_slice)[:, :, 0]
        .astype(np.float32)
        / 255.0
    )


def lpips_distance(
    model: torch.nn.Module,
    reference: np.ndarray,
    reconstruction: np.ndarray,
) -> float:
    """Compute LPIPS for two normalized grayscale images."""

    reference_tensor = (
        torch.from_numpy(
            np.ascontiguousarray(reference)
        )
        .float()
        .unsqueeze(0)
        .unsqueeze(0)
        .repeat(1, 3, 1, 1)
        .cuda()
        * 2.0
        - 1.0
    )

    reconstruction_tensor = (
        torch.from_numpy(
            np.ascontiguousarray(reconstruction)
        )
        .float()
        .unsqueeze(0)
        .unsqueeze(0)
        .repeat(1, 3, 1, 1)
        .cuda()
        * 2.0
        - 1.0
    )

    with torch.no_grad():
        return float(
            model(
                reference_tensor,
                reconstruction_tensor,
            ).item()
        )


def evaluate_canonical_model(
    current_run_root: Path,
    model_input: str,
    lpips_model: torch.nn.Module,
):
    """Render one canonical model and evaluate it against both targets."""

    current_model_root = current_run_root / "model"
    current_dataset_root = current_run_root / "dataset"
    render_root = current_model_root / "render_img"

    shutil.rmtree(
        render_root,
        ignore_errors=True,
    )

    render_command = [
        sys.executable,
        str(MEDGS_REPOSITORY / "render.py"),
        "-s",
        str(current_dataset_root),
        "-m",
        str(current_model_root),
        "--iteration",
        str(ITERATIONS),
        "--poly_degree",
        str(POLY_DEGREE),
        "--camera",
        CAMERA,
        "--pipeline",
        "img",
        "--interp",
        "1",
    ]

    print(f"Rendering {model_input} model:")
    print(shlex.join(render_command))

    subprocess.run(
        render_command,
        cwd=MEDGS_REPOSITORY,
        check=True,
    )

    frame_manifest_df = pd.read_csv(
        current_run_root / "frame_manifest.csv"
    )

    study_tag = str(STUDY_INSTANCE_UID)[-12:]

    prepared_volume_root = (
        PREPARED_4D_ROOT
        / PATIENT_ID
        / f"study_{study_tag}"
        / "volumes"
    )

    raw_volume_path = (
        prepared_volume_root
        / "raw"
        / f"phase_{phase_tag(canonical_phase)}.npy"
    )

    denoised_volume_path = (
        prepared_volume_root
        / "denoised"
        / f"phase_{phase_tag(canonical_phase)}.npy"
    )

    target_volumes = {
        "raw": np.load(
            raw_volume_path,
            mmap_mode="r",
        ),
        "denoised": np.load(
            denoised_volume_path,
            mmap_mode="r",
        ),
    }

    metric_rows = []

    for _, frame in frame_manifest_df.iterrows():
        frame_index = int(frame["FrameIndex"])
        slice_index = int(frame["SliceIndex"])

        reconstruction_path = (
            render_root
            / f"{frame_index:05d}_0.png"
        )

        reconstruction = load_gray_image(
            reconstruction_path
        )

        for evaluation_target, target_volume in (
            target_volumes.items()
        ):
            target = volume_slice_to_gray(
                target_volume,
                slice_index,
            )

            metric_rows.append(
                {
                    "PatientID": PATIENT_ID,
                    "StudyInstanceUID": STUDY_INSTANCE_UID,
                    "PhasePercent": canonical_phase,
                    "ModelInput": model_input,
                    "EvaluationTarget": evaluation_target,
                    "FrameIndex": frame_index,
                    "SliceIndex": slice_index,
                    "SliceCoordinate": float(
                        frame["SliceCoordinate"]
                    ),
                    "PSNR": peak_signal_noise_ratio(
                        target,
                        reconstruction,
                        data_range=1.0,
                    ),
                    "SSIM": structural_similarity(
                        target,
                        reconstruction,
                        data_range=1.0,
                    ),
                    "LPIPS": lpips_distance(
                        lpips_model,
                        target,
                        reconstruction,
                    ),
                    "ReconstructionPath": str(
                        reconstruction_path
                    ),
                }
            )

    metrics_df = pd.DataFrame(metric_rows)

    summary_df = (
        metrics_df.groupby(
            [
                "PatientID",
                "StudyInstanceUID",
                "PhasePercent",
                "ModelInput",
                "EvaluationTarget",
            ],
            as_index=False,
            sort=False,
        )
        .agg(
            Slices=("FrameIndex", "count"),
            PSNR=("PSNR", "mean"),
            SSIM=("SSIM", "mean"),
            LPIPS=("LPIPS", "mean"),
        )
    )

    metrics_path = (
        current_run_root
        / "evaluation_metrics.csv"
    )

    summary_path = (
        current_run_root
        / "evaluation_metrics_summary.csv"
    )

    metrics_df.to_csv(
        metrics_path,
        index=False,
    )

    summary_df.to_csv(
        summary_path,
        index=False,
    )

    print(f"Metrics saved to: {metrics_path}")
    print(f"Summary saved to: {summary_path}")

    return (
        frame_manifest_df,
        metrics_df,
        summary_df,
    )


study_tag = str(STUDY_INSTANCE_UID)[-12:]
phase_name = phase_tag(canonical_phase)

canonical_medgs_root = (
    RESULTS_ROOT
    / "canonical_medgs"
)

raw_run_root = (
    canonical_medgs_root
    / (
        f"patient_{PATIENT_ID}"
        f"__study_{study_tag}"
        f"__phase_{phase_name}"
        f"__raw"
        f"__poly{POLY_DEGREE}"
        f"__iter{ITERATIONS}"
    )
)

denoised_run_root = (
    canonical_medgs_root
    / (
        f"patient_{PATIENT_ID}"
        f"__study_{study_tag}"
        f"__phase_{phase_name}"
        f"__denoised"
        f"__poly{POLY_DEGREE}"
        f"__iter{ITERATIONS}"
    )
)

comparison_lpips_model = (
    lpips.LPIPS(net="alex")
    .cuda()
    .eval()
)

(
    raw_frame_manifest_df,
    raw_metrics_df,
    raw_metrics_summary_df,
) = evaluate_canonical_model(
    raw_run_root,
    model_input="raw",
    lpips_model=comparison_lpips_model,
)

(
    denoised_frame_manifest_df,
    denoised_metrics_df,
    denoised_metrics_summary_df,
) = evaluate_canonical_model(
    denoised_run_root,
    model_input="denoised",
    lpips_model=comparison_lpips_model,
)

del comparison_lpips_model
torch.cuda.empty_cache()

print("Canonical model evaluation completed.")

display(raw_metrics_summary_df)
display(denoised_metrics_summary_df)

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/home/jovyan/shared/mtm_medgs_stack/envs/medgs-worf/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/jovyan/shared/mtm_medgs_stack/envs/medgs-worf/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /home/jovyan/shared/mtm_medgs_stack/envs/medgs-worf/lib/python3.11/site-packages/lpips/weights/v0.1/alex.pth
Rendering raw model:
/home/jovyan/shared/mtm_medgs_stack/envs/medgs-worf/bin/python /home/jovyan/shared/mtm_medgs_stack/repo/MedGS/render.py -s /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__raw__poly2__iter30000/dataset -m /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__raw__poly2__iter30000/model --iteration 30000 --poly_degree 2 --camera mirror --pipeline img --interp 1


/home/jovyan/shared/mtm_medgs_stack/envs/medgs-worf/lib/python3.11/site-packages/lpips/lpips.py:107: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch

Looking for config file in /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__raw__poly2__iter30000/model/cfg_args
Config file found: /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__raw__poly2__iter30000/model/cfg_args
Rendering /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__raw__poly2__iter30000/model
Distance: 1.0 [29/07 14:18:12]
Loading trained model at iteration 30000 [29/07 14:18:12]
Creating Training Transforms [29/07 14:18:12]
Creating Test Transforms [29/07 14:18:18]
AAAAA radius 1.1 [29/07 14:18:21]
AAAAA translate [-0. -0. -0.] center [0. 0. 0.] [29/07 14:18:21]
Generating random point cloud (100000)... [29/07 14:18:21]
Loading Training Cameras [29/07 14:18:21]
Loading Test Cameras [29/07 14:18:22]
Loading checkpoint: /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM

...

Looking for config file in /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoised__poly2__iter30000/model/cfg_args
Config file found: /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoised__poly2__iter30000/model/cfg_args
Rendering /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoised__poly2__iter30000/model
Distance: 1.0 [29/07 14:18:37]
Loading trained model at iteration 30000 [29/07 14:18:37]
Creating Training Transforms [29/07 14:18:37]
Creating Test Transforms [29/07 14:18:43]
AAAAA radius 1.1 [29/07 14:18:46]
AAAAA translate [-0. -0. -0.] center [0. 0. 0.] [29/07 14:18:46]
Generating random point cloud (100000)... [29/07 14:18:46]
Loading Training Cameras [29/07 14:18:46]
Loading Test Cameras [29/07 14:18:46]
Loading checkpoint: /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs

Rendering progress: 100%|██████████| 147/147 [00:05<00:00, 26.21it/s]


Metrics saved to: /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoised__poly2__iter30000/evaluation_metrics.csv
Summary saved to: /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__denoised__poly2__iter30000/evaluation_metrics_summary.csv
Canonical model evaluation completed.


,PatientID,StudyInstanceUID,PhasePercent,ModelInput,EvaluationTarget,Slices,PSNR,SSIM,LPIPS
0,117_HM10395,1.3.6.1.4.1.14519.5.2.1.6834.5010.378204929111417980831212264180,20.0,raw,raw,147,37.876642,0.938719,0.086408
1,117_HM10395,1.3.6.1.4.1.14519.5.2.1.6834.5010.378204929111417980831212264180,20.0,raw,denoised,147,35.564167,0.950572,0.069546


,PatientID,StudyInstanceUID,PhasePercent,ModelInput,EvaluationTarget,Slices,PSNR,SSIM,LPIPS
0,117_HM10395,1.3.6.1.4.1.14519.5.2.1.6834.5010.378204929111417980831212264180,20.0,denoised,raw,147,33.486089,0.886973,0.177803
1,117_HM10395,1.3.6.1.4.1.14519.5.2.1.6834.5010.378204929111417980831212264180,20.0,denoised,denoised,147,40.621407,0.962294,0.057387


In [36]:
canonical_comparison_df = pd.concat(
    [
        raw_metrics_summary_df,
        denoised_metrics_summary_df,
    ],
    ignore_index=True,
)

canonical_comparison_df = (
    canonical_comparison_df.sort_values(
        [
            "EvaluationTarget",
            "ModelInput",
        ]
    )
    .reset_index(drop=True)
)

raw_target_comparison_df = (
    canonical_comparison_df.loc[
        canonical_comparison_df[
            "EvaluationTarget"
        ]
        == "raw"
    ]
    .copy()
    .reset_index(drop=True)
)

raw_model_reference = (
    raw_target_comparison_df.loc[
        raw_target_comparison_df["ModelInput"]
        == "raw"
    ]
    .iloc[0]
)

raw_target_comparison_df[
    "PSNR_change_vs_raw_model"
] = (
    raw_target_comparison_df["PSNR"]
    - raw_model_reference["PSNR"]
)

raw_target_comparison_df[
    "SSIM_change_vs_raw_model"
] = (
    raw_target_comparison_df["SSIM"]
    - raw_model_reference["SSIM"]
)

raw_target_comparison_df[
    "LPIPS_change_vs_raw_model"
] = (
    raw_target_comparison_df["LPIPS"]
    - raw_model_reference["LPIPS"]
)

comparison_path = (
    canonical_medgs_root
    / (
        f"patient_{PATIENT_ID}"
        f"__study_{study_tag}"
        f"__phase_{phase_name}"
        f"__raw_vs_denoised_summary.csv"
    )
)

canonical_comparison_df.to_csv(
    comparison_path,
    index=False,
)

print(f"Comparison saved to: {comparison_path}")
print(
    "Fair model comparison uses EvaluationTarget = raw. "
    "Higher PSNR/SSIM and lower LPIPS are better."
)

display(canonical_comparison_df)
display(raw_target_comparison_df)

Comparison saved to: /home/jovyan/shared/mtm_medgs_stack/results/canonical_medgs/patient_117_HM10395__study_831212264180__phase_20__raw_vs_denoised_summary.csv
Fair model comparison uses EvaluationTarget = raw. Higher PSNR/SSIM and lower LPIPS are better.


,PatientID,StudyInstanceUID,PhasePercent,ModelInput,EvaluationTarget,Slices,PSNR,SSIM,LPIPS
0,117_HM10395,1.3.6.1.4.1.14519.5.2.1.6834.5010.378204929111417980831212264180,20.0,denoised,denoised,147,40.621407,0.962294,0.057387
1,117_HM10395,1.3.6.1.4.1.14519.5.2.1.6834.5010.378204929111417980831212264180,20.0,raw,denoised,147,35.564167,0.950572,0.069546
2,117_HM10395,1.3.6.1.4.1.14519.5.2.1.6834.5010.378204929111417980831212264180,20.0,denoised,raw,147,33.486089,0.886973,0.177803
3,117_HM10395,1.3.6.1.4.1.14519.5.2.1.6834.5010.378204929111417980831212264180,20.0,raw,raw,147,37.876642,0.938719,0.086408


,PatientID,StudyInstanceUID,PhasePercent,ModelInput,EvaluationTarget,Slices,PSNR,SSIM,LPIPS,PSNR_change_vs_raw_model,SSIM_change_vs_raw_model,LPIPS_change_vs_raw_model
0,117_HM10395,1.3.6.1.4.1.14519.5.2.1.6834.5010.378204929111417980831212264180,20.0,denoised,raw,147,33.486089,0.886973,0.177803,-4.390553,-0.051746,0.091395
1,117_HM10395,1.3.6.1.4.1.14519.5.2.1.6834.5010.378204929111417980831212264180,20.0,raw,raw,147,37.876642,0.938719,0.086408,0.000000,0.000000,0.000000
